# Gerador de Specs em CSV — PNAD COVID Parquet

Este notebook foi gerado a partir do arquivo de referência `refined_analysis_pnad(1).ipynb` e das queries anexadas.

**Objetivo:** ler as tabelas Parquet locais da pasta do projeto, executar as consultas analíticas e gravar um arquivo CSV de resultado/spec para cada query.

Estrutura esperada no projeto:

```text
.
├── este_notebook.ipynb
├── output/
│   ├── pnad_covid/
│   │   ├── base_saude/
│   │   ├── base_comportamento/
│   │   ├── base_economico/
│   │   ├── dim_perfil/
│   │   └── dim_localizacao/
│   └── specs/
```

> Se sua pasta for diferente, altere a variável `BASE_PATH` na célula de configuração.


In [7]:
#!pip install duckdb pandas pyarrow fastparquet

In [8]:
# Se precisar instalar no ambiente local, descomente:
# %pip install duckdb pandas pyarrow

import os
import re
import json
import warnings
from pathlib import Path

import duckdb
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)


In [10]:
# =========================================================
# CONFIGURAÇÃO LOCAL
# =========================================================

# Pasta onde estão as tabelas Parquet.
# Exemplo Windows:
# BASE_PATH = r"C:\Users\renat\OneDrive\Área de Trabalho\TC\FASE3\Tech Challenge\output\pnad_covid"
BASE_PATH = Path("../output/pnad_covid")

# Pasta onde os CSVs/specs serão gravados
SPECS_PATH = Path("../output/specs")
SPECS_PATH.mkdir(parents=True, exist_ok=True)

# Separador CSV:
# - use ";" para abrir melhor no Excel pt-BR
# - use "," se preferir padrão CSV tradicional
CSV_SEP = ";"

print("BASE_PATH :", BASE_PATH.resolve())
print("SPECS_PATH:", SPECS_PATH.resolve())


BASE_PATH : C:\Users\renat\OneDrive\Área de Trabalho\Tech Challenge\output\pnad_covid
SPECS_PATH: C:\Users\renat\OneDrive\Área de Trabalho\Tech Challenge\output\specs


In [11]:
# =========================================================
# CONEXÃO DUCKDB + REGISTRO DAS VIEWS SOBRE PARQUET
# =========================================================

con = duckdb.connect(database=":memory:")

def parquet_pattern(table_name: str) -> str:
    """Retorna padrão recursivo para ler todos os parquets da tabela."""
    table_dir = BASE_PATH / table_name
    return str(table_dir / "**" / "*.parquet").replace("\\", "/")

def validate_table_path(table_name: str) -> None:
    table_dir = BASE_PATH / table_name
    if not table_dir.exists():
        raise FileNotFoundError(
            f"Não encontrei a pasta da tabela: {table_dir}\n"
            f"Confira a variável BASE_PATH ou o nome da tabela."
        )

tables = [
    "base_saude",
    "base_comportamento",
    "base_economico",
    "dim_perfil",
    "dim_localizacao",
]

for table in tables:
    validate_table_path(table)
    pattern = parquet_pattern(table)
    con.execute(f'''
        CREATE OR REPLACE VIEW {table} AS
        SELECT *
        FROM read_parquet('{pattern}', hive_partitioning=true, union_by_name=true)
    ''')
    qtd = con.execute(f"SELECT COUNT(*) AS qtd FROM {table}").fetchone()[0]
    print(f"{table:22s} -> {qtd:,} registros")

print("\nViews criadas com sucesso.")


base_saude             -> 1,149,197 registros
base_comportamento     -> 1,149,197 registros
base_economico         -> 1,149,197 registros
dim_perfil             -> 1,149,197 registros
dim_localizacao        -> 1,149,197 registros

Views criadas com sucesso.


In [12]:
# =========================================================
# QUERIES ANALÍTICAS
# =========================================================
# Observação:
# As queries originais referenciavam db_pnad_covid.tabela.
# A função normalize_sql() remove esse prefixo, pois aqui as tabelas são views locais do DuckDB.

QUERIES = [('01_evolucao_geral_sintomas_gripais', 'Evolução geral de sintomas gripais por mês', 'SELECT\n    mes_ref,\n    ROUND(\n        SUM(ind_teve_sintoma_gripal * peso_amostral) / SUM(peso_amostral) * 100,\n        2\n    ) AS pct_populacao_com_sintoma_gripal\nFROM db_pnad_covid.base_saude\nGROUP BY mes_ref\nORDER BY mes_ref'), ('02_sintomas_gripais_por_regiao', 'Sintomas gripais por região', 'SELECT\n    s.mes_ref,\n    l.regiao,\n    ROUND(\n        SUM(s.ind_teve_sintoma_gripal * s.peso_amostral) / SUM(s.peso_amostral) * 100,\n        2\n    ) AS pct_sintoma_gripal\nFROM db_pnad_covid.base_saude s\nJOIN db_pnad_covid.dim_localizacao l\n    ON s.uf = l.uf\n   AND s.id_domicilio = l.id_domicilio\n   AND s.id_morador = l.id_morador\n   AND s.mes_entrevista = l.mes_entrevista\n   AND s.mes_ref = l.mes_ref\nGROUP BY s.mes_ref, l.regiao\nORDER BY s.mes_ref, pct_sintoma_gripal DESC'), ('03_sintomas_por_faixa_etaria', 'Sintomas por faixa etária', 'SELECT\n    s.mes_ref,\n    p.faixa_etaria,\n    ROUND(\n        SUM(s.ind_teve_sintoma_gripal * s.peso_amostral) / SUM(s.peso_amostral) * 100,\n        2\n    ) AS pct_sintoma_gripal\nFROM db_pnad_covid.base_saude s\nJOIN db_pnad_covid.dim_perfil p\n    ON s.uf = p.uf\n   AND s.id_domicilio = p.id_domicilio\n   AND s.id_morador = p.id_morador\n   AND s.mes_entrevista = p.mes_entrevista\n   AND s.mes_ref = p.mes_ref\nGROUP BY s.mes_ref, p.faixa_etaria\nORDER BY s.mes_ref, p.faixa_etaria'), ('04_gravidade_dificuldade_respirar', 'Gravidade clínica: dificuldade para respirar', "SELECT\n    mes_ref,\n    ROUND(\n        SUM(CASE WHEN sintoma_dificuldade_respirar = 'Sim' THEN peso_amostral ELSE 0 END)\n        / SUM(peso_amostral) * 100,\n        2\n    ) AS pct_dificuldade_respirar\nFROM db_pnad_covid.base_saude\nGROUP BY mes_ref\nORDER BY mes_ref"), ('05_sintomas_com_comorbidades', 'População com sintomas e comorbidades', "SELECT\n    mes_ref,\n    ROUND(\n        SUM(\n            CASE \n                WHEN ind_teve_sintoma_gripal = 1\n                 AND (\n                    medico_deu_diagnostico_diabetes = 'Sim'\n                    OR medico_deu_diagnostico_hipertensao = 'Sim'\n                    OR medico_deu_diagnostico_doencas_coracao_infarto_angina = 'Sim'\n                    OR medico_deu_diagnostico_asma_bronquite_enfisema_doencas = 'Sim'\n                 )\n                THEN peso_amostral ELSE 0 \n            END\n        ) / SUM(peso_amostral) * 100,\n        2\n    ) AS pct_sintomaticos_com_comorbidade\nFROM db_pnad_covid.base_saude\nGROUP BY mes_ref\nORDER BY mes_ref"), ('06_busca_atendimento_saude', 'Busca por atendimento de saúde', "SELECT\n    mes_ref,\n    ROUND(\n        SUM(CASE WHEN buscou_atendimento_saude = 'Sim' THEN peso_amostral ELSE 0 END)\n        / SUM(peso_amostral) * 100,\n        2\n    ) AS pct_buscou_atendimento\nFROM db_pnad_covid.base_comportamento\nGROUP BY mes_ref\nORDER BY mes_ref"), ('07_local_atendimento', 'Onde a população buscou atendimento', "SELECT\n    mes_ref,\n    ROUND(SUM(CASE WHEN atendimento_posto_ubs = 'Sim' THEN peso_amostral ELSE 0 END) / SUM(peso_amostral) * 100, 2) AS pct_ubs,\n    ROUND(SUM(CASE WHEN atendimento_ps_publico = 'Sim' THEN peso_amostral ELSE 0 END) / SUM(peso_amostral) * 100, 2) AS pct_ps_publico,\n    ROUND(SUM(CASE WHEN atendimento_hospital_publico = 'Sim' THEN peso_amostral ELSE 0 END) / SUM(peso_amostral) * 100, 2) AS pct_hospital_publico,\n    ROUND(SUM(CASE WHEN atendimento_ps_privado = 'Sim' THEN peso_amostral ELSE 0 END) / SUM(peso_amostral) * 100, 2) AS pct_ps_privado,\n    ROUND(SUM(CASE WHEN atendimento_hospital_privado = 'Sim' THEN peso_amostral ELSE 0 END) / SUM(peso_amostral) * 100, 2) AS pct_hospital_privado\nFROM db_pnad_covid.base_comportamento\nGROUP BY mes_ref\nORDER BY mes_ref"), ('08_uso_mascara_por_mes', 'Uso de máscara por mês', "SELECT\n    mes_ref,\n    ROUND(\n        SUM(CASE WHEN usa_mascara = 'Sim' THEN peso_amostral ELSE 0 END)\n        / SUM(peso_amostral) * 100,\n        2\n    ) AS pct_usa_mascara\nFROM db_pnad_covid.base_comportamento\nGROUP BY mes_ref\nORDER BY mes_ref"), ('09_home_office_por_regiao', 'Home office por região', "SELECT\n    c.mes_ref,\n    l.regiao,\n    ROUND(\n        SUM(CASE WHEN c.fez_home_office = 'Sim' THEN c.peso_amostral ELSE 0 END)\n        / SUM(c.peso_amostral) * 100,\n        2\n    ) AS pct_home_office\nFROM db_pnad_covid.base_comportamento c\nJOIN db_pnad_covid.dim_localizacao l\n    ON c.uf = l.uf\n   AND c.id_domicilio = l.id_domicilio\n   AND c.id_morador = l.id_morador\n   AND c.mes_entrevista = l.mes_entrevista\n   AND c.mes_ref = l.mes_ref\nGROUP BY c.mes_ref, l.regiao\nORDER BY c.mes_ref, pct_home_office DESC"), ('10_situacao_mercado_trabalho', 'Situação do mercado de trabalho', 'SELECT\n    mes_ref,\n    situacao_mercado_trabalho,\n    ROUND(SUM(peso_amostral), 0) AS populacao_estimada\nFROM db_pnad_covid.base_comportamento\nGROUP BY mes_ref, situacao_mercado_trabalho\nORDER BY mes_ref, populacao_estimada DESC'), ('11_dependencia_auxilio_emergencial', 'Dependência de auxílio emergencial', "SELECT\n    mes_ref,\n    ROUND(\n        SUM(CASE WHEN recebeu_auxilio_emergencial = 'Sim' THEN peso_amostral ELSE 0 END)\n        / SUM(peso_amostral) * 100,\n        2\n    ) AS pct_recebeu_auxilio,\n    ROUND(\n        SUM(CASE WHEN recebeu_auxilio_emergencial = 'Sim' THEN valor_auxilio_emergencial * peso_amostral ELSE 0 END)\n        / NULLIF(SUM(CASE WHEN recebeu_auxilio_emergencial = 'Sim' THEN peso_amostral ELSE 0 END), 0),\n        2\n    ) AS valor_medio_auxilio\nFROM db_pnad_covid.base_economico\nGROUP BY mes_ref\nORDER BY mes_ref"), ('12_renda_efetiva_media_por_regiao', 'Renda efetiva média por região', 'SELECT\n    e.mes_ref,\n    l.regiao,\n    ROUND(\n        SUM(e.renda_efetiva_trabalho_principal * e.peso_amostral)\n        / NULLIF(SUM(CASE WHEN e.renda_efetiva_trabalho_principal IS NOT NULL THEN e.peso_amostral ELSE 0 END), 0),\n        2\n    ) AS renda_efetiva_media\nFROM db_pnad_covid.base_economico e\nJOIN db_pnad_covid.dim_localizacao l\n    ON e.uf = l.uf\n   AND e.id_domicilio = l.id_domicilio\n   AND e.id_morador = l.id_morador\n   AND e.mes_entrevista = l.mes_entrevista\n   AND e.mes_ref = l.mes_ref\nGROUP BY e.mes_ref, l.regiao\nORDER BY e.mes_ref, renda_efetiva_media'), ('13_insights_prevencao_proximo_surto', 'Insights para prevenção de próximo surto de COVID', "WITH loc AS (\n    SELECT\n        uf,\n        id_domicilio,\n        id_morador,\n        mes_entrevista,\n        mes_ref,\n        regiao\n    FROM db_pnad_covid.dim_localizacao\n),\n\nsaude AS (\n    SELECT\n        s.mes_ref,\n        l.regiao,\n        SUM(s.peso_amostral) AS peso_total_saude,\n        SUM(s.ind_teve_sintoma_gripal * s.peso_amostral) AS peso_sintomas,\n        SUM(\n            CASE \n                WHEN s.sintoma_dificuldade_respirar = 'Sim' \n                THEN s.peso_amostral \n                ELSE 0 \n            END\n        ) AS peso_dificuldade_respirar\n    FROM db_pnad_covid.base_saude s\n    JOIN loc l\n        ON s.uf = l.uf\n       AND s.id_domicilio = l.id_domicilio\n       AND s.id_morador = l.id_morador\n       AND s.mes_entrevista = l.mes_entrevista\n       AND s.mes_ref = l.mes_ref\n    GROUP BY s.mes_ref, l.regiao\n),\n\ncomportamento AS (\n    SELECT\n        c.mes_ref,\n        l.regiao,\n        SUM(c.peso_amostral) AS peso_total_comportamento,\n        SUM(\n            CASE \n                WHEN c.fez_home_office = 'Sim' \n                THEN c.peso_amostral \n                ELSE 0 \n            END\n        ) AS peso_home_office\n    FROM db_pnad_covid.base_comportamento c\n    JOIN loc l\n        ON c.uf = l.uf\n       AND c.id_domicilio = l.id_domicilio\n       AND c.id_morador = l.id_morador\n       AND c.mes_entrevista = l.mes_entrevista\n       AND c.mes_ref = l.mes_ref\n    GROUP BY c.mes_ref, l.regiao\n),\n\neconomico AS (\n    SELECT\n        e.mes_ref,\n        l.regiao,\n        SUM(e.peso_amostral) AS peso_total_economico,\n        SUM(\n            CASE \n                WHEN e.recebeu_auxilio_emergencial = 'Sim' \n                THEN e.peso_amostral \n                ELSE 0 \n            END\n        ) AS peso_auxilio\n    FROM db_pnad_covid.base_economico e\n    JOIN loc l\n        ON e.uf = l.uf\n       AND e.id_domicilio = l.id_domicilio\n       AND e.id_morador = l.id_morador\n       AND e.mes_entrevista = l.mes_entrevista\n       AND e.mes_ref = l.mes_ref\n    GROUP BY e.mes_ref, l.regiao\n)\n\nSELECT\n    s.mes_ref,\n    s.regiao,\n\n    ROUND(s.peso_sintomas / s.peso_total_saude * 100, 2) AS pct_sintomas,\n\n    ROUND(s.peso_dificuldade_respirar / s.peso_total_saude * 100, 2) AS pct_dificuldade_respirar,\n\n    ROUND(c.peso_home_office / c.peso_total_comportamento * 100, 2) AS pct_home_office,\n\n    ROUND(e.peso_auxilio / e.peso_total_economico * 100, 2) AS pct_auxilio_emergencial\n\nFROM saude s\nLEFT JOIN comportamento c\n    ON s.mes_ref = c.mes_ref\n   AND s.regiao = c.regiao\nLEFT JOIN economico e\n    ON s.mes_ref = e.mes_ref\n   AND s.regiao = e.regiao\nORDER BY s.mes_ref, pct_sintomas DESC")]

def normalize_sql(sql: str) -> str:
    sql = sql.strip().rstrip(";")
    sql = sql.replace("db_pnad_covid.", "")
    return sql

print(f"Total de queries carregadas: {len(QUERIES)}")
for slug, titulo, _ in QUERIES:
    print(f"- {slug} | {titulo}")


Total de queries carregadas: 13
- 01_evolucao_geral_sintomas_gripais | Evolução geral de sintomas gripais por mês
- 02_sintomas_gripais_por_regiao | Sintomas gripais por região
- 03_sintomas_por_faixa_etaria | Sintomas por faixa etária
- 04_gravidade_dificuldade_respirar | Gravidade clínica: dificuldade para respirar
- 05_sintomas_com_comorbidades | População com sintomas e comorbidades
- 06_busca_atendimento_saude | Busca por atendimento de saúde
- 07_local_atendimento | Onde a população buscou atendimento
- 08_uso_mascara_por_mes | Uso de máscara por mês
- 09_home_office_por_regiao | Home office por região
- 10_situacao_mercado_trabalho | Situação do mercado de trabalho
- 11_dependencia_auxilio_emergencial | Dependência de auxílio emergencial
- 12_renda_efetiva_media_por_regiao | Renda efetiva média por região
- 13_insights_prevencao_proximo_surto | Insights para prevenção de próximo surto de COVID


In [ ]:
# =========================================================
# EXECUTA CADA QUERY E GERA UM CSV EM output/specs
# =========================================================

resultados = []

for slug, titulo, sql in QUERIES:
    print("=" * 100)
    print(f"Executando: {slug} | {titulo}")

    sql_local = normalize_sql(sql)

    try:
        df = con.execute(sql_local).df()

        output_file = SPECS_PATH / f"{slug}.csv"
        df.to_csv(output_file, index=False, sep=CSV_SEP, encoding="utf-8-sig")

        resultados.append({
            "slug": slug,
            "titulo": titulo,
            "arquivo_csv": str(output_file).replace("\\", "/"),
            "linhas": len(df),
            "colunas": len(df.columns),
            "status": "OK",
            "erro": ""
        })

        print(f"OK -> {output_file} | linhas: {len(df)} | colunas: {len(df.columns)}")
        display(df.head(10))

    except Exception as e:
        resultados.append({
            "slug": slug,
            "titulo": titulo,
            "arquivo_csv": "",
            "linhas": 0,
            "colunas": 0,
            "status": "ERRO",
            "erro": str(e)
        })
        print(f"ERRO -> {slug}")
        print(e)

resumo_specs = pd.DataFrame(resultados)
resumo_specs


Executando: 01_evolucao_geral_sintomas_gripais | Evolução geral de sintomas gripais por mês
OK -> ..\output\specs\01_evolucao_geral_sintomas_gripais.csv | linhas: 3 | colunas: 2


,MES_REF,pct_populacao_com_sintoma_gripal
0,2020-09,2.36
1,2020-10,2.03
2,2020-11,2.24


Executando: 02_sintomas_gripais_por_regiao | Sintomas gripais por região
OK -> ..\output\specs\02_sintomas_gripais_por_regiao.csv | linhas: 15 | colunas: 3


,MES_REF,regiao,pct_sintoma_gripal
0,2020-09,Centro-Oeste,3.30
1,2020-09,Norte,3.30
2,2020-09,Sul,2.85
3,2020-09,Sudeste,2.42
4,2020-09,Nordeste,2.38
5,2020-10,Norte,3.15
6,2020-10,Centro-Oeste,2.67
7,2020-10,Sul,2.36
8,2020-10,Nordeste,2.10
9,2020-10,Sudeste,2.01


Executando: 03_sintomas_por_faixa_etaria | Sintomas por faixa etária
OK -> ..\output\specs\03_sintomas_por_faixa_etaria.csv | linhas: 18 | colunas: 3


...

Executando: 12_renda_efetiva_media_por_regiao | Renda efetiva média por região
OK -> ..\output\specs\12_renda_efetiva_media_por_regiao.csv | linhas: 15 | colunas: 3


,MES_REF,regiao,renda_efetiva_media
0,2020-09,Nordeste,1612.75
1,2020-09,Norte,1674.36
2,2020-09,Centro-Oeste,2345.36
3,2020-09,Sul,2431.50
4,2020-09,Sudeste,2521.89
5,2020-10,Nordeste,1621.67
6,2020-10,Norte,1735.53
7,2020-10,Centro-Oeste,2371.26
8,2020-10,Sul,2448.39
9,2020-10,Sudeste,2552.52


Executando: 13_insights_prevencao_proximo_surto | Insights para prevenção de próximo surto de COVID
OK -> ..\output\specs\13_insights_prevencao_proximo_surto.csv | linhas: 15 | colunas: 6


,MES_REF,regiao,pct_sintomas,pct_dificuldade_respirar,pct_home_office,pct_auxilio_emergencial
0,2020-09,Norte,3.30,0.65,1.57,64.83
1,2020-09,Centro-Oeste,3.30,0.93,3.86,45.86
2,2020-09,Sul,2.85,0.63,4.57,32.97
3,2020-09,Sudeste,2.42,0.57,6.14,40.06
4,2020-09,Nordeste,2.38,0.35,2.57,63.14
5,2020-10,Norte,3.15,0.57,1.55,63.39
6,2020-10,Centro-Oeste,2.67,0.55,3.76,44.62
7,2020-10,Sul,2.36,0.45,4.19,31.98
8,2020-10,Nordeste,2.10,0.38,2.42,60.97
9,2020-10,Sudeste,2.01,0.50,5.79,38.80


,slug,titulo,arquivo_csv,linhas,colunas,status,erro
0,01_evolucao_geral_sintomas_gripais,Evolução geral de sintomas gripais por mês,../output/specs/01_evolucao_geral_sintomas_gri...,3,2,OK,
1,02_sintomas_gripais_por_regiao,Sintomas gripais por região,../output/specs/02_sintomas_gripais_por_regiao...,15,3,OK,
2,03_sintomas_por_faixa_etaria,Sintomas por faixa etária,../output/specs/03_sintomas_por_faixa_etaria.csv,18,3,OK,
3,04_gravidade_dificuldade_respirar,Gravidade clínica: dificuldade para respirar,../output/specs/04_gravidade_dificuldade_respi...,3,2,OK,
4,05_sintomas_com_comorbidades,População com sintomas e comorbidades,../output/specs/05_sintomas_com_comorbidades.csv,3,2,OK,
5,06_busca_atendimento_saude,Busca por atendimento de saúde,../output/specs/06_busca_atendimento_saude.csv,3,2,OK,
6,07_local_atendimento,Onde a população buscou atendimento,,0,0,ERRO,"Binder Error: Referenced column ""atendimento_p..."
7,08_uso_mascara_por_mes,Uso de máscara por mês,,0,0,ERRO,"Binder Error: Referenced column ""usa_mascara"" ..."
8,09_home_office_por_regiao,Home office por região,../output/specs/09_home_office_por_regiao.csv,15,3,OK,
9,10_situacao_mercado_trabalho,Situação do mercado de trabalho,../output/specs/10_situacao_mercado_trabalho.csv,15,3,OK,


In [14]:
# =========================================================
# GERA ÍNDICE/RESUMO DOS SPECS
# =========================================================

indice_file = SPECS_PATH / "_indice_specs.csv"
resumo_specs.to_csv(indice_file, index=False, sep=CSV_SEP, encoding="utf-8-sig")

print(f"Índice gerado em: {indice_file}")
display(resumo_specs)


Índice gerado em: ..\output\specs\_indice_specs.csv


,slug,titulo,arquivo_csv,linhas,colunas,status,erro
0,01_evolucao_geral_sintomas_gripais,Evolução geral de sintomas gripais por mês,../output/specs/01_evolucao_geral_sintomas_gri...,3,2,OK,
1,02_sintomas_gripais_por_regiao,Sintomas gripais por região,../output/specs/02_sintomas_gripais_por_regiao...,15,3,OK,
2,03_sintomas_por_faixa_etaria,Sintomas por faixa etária,../output/specs/03_sintomas_por_faixa_etaria.csv,18,3,OK,
3,04_gravidade_dificuldade_respirar,Gravidade clínica: dificuldade para respirar,../output/specs/04_gravidade_dificuldade_respi...,3,2,OK,
4,05_sintomas_com_comorbidades,População com sintomas e comorbidades,../output/specs/05_sintomas_com_comorbidades.csv,3,2,OK,
5,06_busca_atendimento_saude,Busca por atendimento de saúde,../output/specs/06_busca_atendimento_saude.csv,3,2,OK,
6,07_local_atendimento,Onde a população buscou atendimento,,0,0,ERRO,"Binder Error: Referenced column ""atendimento_p..."
7,08_uso_mascara_por_mes,Uso de máscara por mês,,0,0,ERRO,"Binder Error: Referenced column ""usa_mascara"" ..."
8,09_home_office_por_regiao,Home office por região,../output/specs/09_home_office_por_regiao.csv,15,3,OK,
9,10_situacao_mercado_trabalho,Situação do mercado de trabalho,../output/specs/10_situacao_mercado_trabalho.csv,15,3,OK,


In [15]:
# =========================================================
# VALIDAÇÃO FINAL
# =========================================================

erros = resumo_specs[resumo_specs["status"] != "OK"]

if len(erros) == 0:
    print("Todos os specs foram gerados com sucesso!")
    print(f"Arquivos disponíveis em: {SPECS_PATH.resolve()}")
else:
    print(f"Foram encontrados {len(erros)} erro(s). Veja abaixo:")
    display(erros[["slug", "titulo", "erro"]])


Foram encontrados 2 erro(s). Veja abaixo:


,slug,titulo,erro
6,07_local_atendimento,Onde a população buscou atendimento,"Binder Error: Referenced column ""atendimento_p..."
7,08_uso_mascara_por_mes,Uso de máscara por mês,"Binder Error: Referenced column ""usa_mascara"" ..."


## Observações

- O notebook usa **DuckDB** para consultar diretamente os arquivos Parquet locais, sem depender do Athena.
- Os arquivos são salvos em `output/specs`.
- O arquivo `_indice_specs.csv` lista todos os specs gerados, quantidade de linhas, colunas e status.
- Caso alguma coluna tenha nome diferente no seu Parquet, a célula de execução mostrará o erro exato da query para ajuste.
